In [ ]:
# ======================================
# SmartChat Insight
#  Módulo de Predicción y Recomendación
# ======================================

# Importar librerías
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:

# -------------------------------
# 1. Cargar los archivos finales
# -------------------------------
clientes = pd.read_csv("../data/outputs/final/clientes_powerbi.csv", sep=";")
clientes_final = pd.read_csv("../data/outputs/final/clientes_final_powerbi.csv", sep=";")
productos = pd.read_csv("../data/outputs/final/productos_powerbi.csv", sep=";")
resumen = pd.read_csv("../data/outputs/final/resumen_clientes.csv", sep=";")


In [ ]:
# -------------------------------
# 2. Preparar datos para predicción
# -------------------------------
# Cantidad de mensajes, días sin respuesta
clientes["mensajes"] = clientes["mensajes"].fillna(0)
clientes["dias_desde_ultimo"] = clientes["dias_desde_ultimo"].fillna(0)

# Etiqueta objetivo: 1 si cliente activo/frecuente, 0 si no
clientes["activo"] = clientes["estado"].apply(lambda x: 1 if str(x).lower() in ["frecuente", "activo"] else 0)

X = clientes[["mensajes", "dias_desde_ultimo"]]
y = clientes["activo"]


In [ ]:
# -------------------------------
# 3. Entrenar modelo
# -------------------------------
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

modelo = DecisionTreeClassifier(max_depth=4, random_state=42)
modelo.fit(X_train, y_train)

print("\n=== EVALUACIÓN DEL MODELO ===")
y_pred = modelo.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
# -------------------------------
# 4. Generar probabilidades y alertas
# -------------------------------
clientes["probabilidad_activo"] = modelo.predict_proba(X)[:, 1]
clientes["probabilidad_abandono"] = 1 - clientes["probabilidad_activo"]

def clasificar_riesgo(p):
    if p >= 0.7:
        return "Bajo riesgo"
    elif p >= 0.4:
        return "Riesgo medio"
    else:
        return "Alto riesgo"

clientes["nivel_riesgo"] = clientes["probabilidad_activo"].apply(clasificar_riesgo)

clientes_riesgo = clientes[clientes["nivel_riesgo"] == "Alto riesgo"]

print(f"\nClientes en alto riesgo detectados: {len(clientes_riesgo)}")


In [ ]:
# -------------------------------
# 5. Guardar resultados para Power BI
# -------------------------------
clientes_resultados = clientes[
    ["user", "mensajes", "dias_desde_ultimo", "estado", 
     "probabilidad_activo", "nivel_riesgo"]
]
clientes_resultados.to_csv("../data/outputs/final/clientes_predicciones.csv", index=False, sep=";")


In [ ]:
# -------------------------------
# 6. Sistema de recomendación
# -------------------------------
# Matriz cliente-producto basada en menciones o interacciones
if "user" in productos.columns and "producto" in productos.columns:
    matriz = pd.pivot_table(productos, index="user", columns="producto", values="menciones", fill_value=0)
    similitud = cosine_similarity(matriz)
    similaridad_df = pd.DataFrame(similitud, index=matriz.index, columns=matriz.index)

    # Recomendaciones de clientes similares
    cliente_demo = matriz.index[0]
    recomendados = similaridad_df[cliente_demo].sort_values(ascending=False).head(5)
    print(f"\n💡 Clientes más similares a {cliente_demo}:")
    print(recomendados)

    # Exportar matriz de similitud
    similaridad_df.to_csv("../data/outputs/final/clientes_similitud.csv", sep=";")
else:
    print("\nNo se encontraron columnas 'user' o 'producto' en productos_powerbi.csv.")

# ---------------------------------------------------
# 7. Alertas automáticas para acciones de WhatsApp
# ---------------------------------------------------
print("\n=== ALERTAS AUTOMÁTICAS ===")
for _, row in clientes_riesgo.iterrows():
    mensaje = (
        f"⚠️ Alerta: Cliente {row['user']} tiene alta probabilidad de abandono "
        f"({row['probabilidad_abandono']:.2f}). Enviar mensaje personalizado."
    )
    print(mensaje)

print("\n Archivos generados:")
print("- clientes_predicciones.csv")
print("- clientes_similitud.csv (si aplica)")



=== EVALUACIÓN DEL MODELO ===
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        51
           1       1.00      1.00      1.00        12

    accuracy                           1.00        63
   macro avg       1.00      1.00      1.00        63
weighted avg       1.00      1.00      1.00        63


🚨 Clientes en alto riesgo detectados: 230

⚠️ No se encontraron columnas 'user' o 'producto' en productos_powerbi.csv.

=== ALERTAS AUTOMÁTICAS ===
⚠️ Alerta: Cliente +51 924 523 653 tiene alta probabilidad de abandono (1.00). Enviar mensaje personalizado.
⚠️ Alerta: Cliente +57 300 2258585 tiene alta probabilidad de abandono (1.00). Enviar mensaje personalizado.
⚠️ Alerta: Cliente +57 300 3542763 tiene alta probabilidad de abandono (1.00). Enviar mensaje personalizado.
⚠️ Alerta: Cliente +57 300 5157021 tiene alta probabilidad de abandono (1.00). Enviar mensaje personalizado.
⚠️ Alerta: Cliente +57 300 5215647 tiene alta probabilida